# Offline Data Drift Report

Минимальный воспроизводимый сценарий офлайн-анализа. Мы создадим две выборки – reference и current, внесём в current известные изменения и посмотрим, как они отражаются в отчёте.

После подготовки данных основной путь состоит из трёх шагов:

1. Создать анализатор и профиль reference-данных.
2. Сравнить current-данные с reference-профилем.
3. Сгенерировать и открыть HTML-отчёт.

Adversarial Validation – дополнительная проверка того, насколько хорошо можно различить две выборки.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
import rich

# Генератор демо-данных с заранее заданными изменениями
from demo_utils import build_offline_demo_data

# Офлайн-анализ и настройка конфигурации
from drift_guardian.analyzer.offline.offline_mode import OfflineWrapper
from drift_guardian.config_handler.auto_config_builder import ConfigBuildOptions

# Генерация HTML-отчёта и его показ внутри ноутбука
from drift_guardian.reporting import display_html_report, generate_html_report

---

### Подготовка данных: reference и current

`build_offline_demo_data` создаёт две воспроизводимые выборки: reference служит базой для сравнения, а в current внесены контролируемые изменения:

- `age` – распределение остаётся стабильным;
- `income` – распределение немного смещается, доля пропусков растёт; при текущих порогах ожидается статус `warning`;
- `country` – появляется новая категория `NEW_COUNTRY`;
- `prediction_score` – меняется распределение предсказаний.

Ниже посмотрим на первые строки выборок и сравним несколько показателей до запуска анализатора.

In [2]:
# генерация демо-данных
reference_df, current_df = build_offline_demo_data(
    reference_rows=10_000,
    current_rows=2_000,
    seed=42,
)
print(f"Reference: {reference_df.shape}")
print(f"Current:   {current_df.shape}")
display(reference_df.head())
display(current_df.head())

Reference: (10000, 4)
Current:   (2000, 4)


,age,income,country,prediction_score
0,43.0,69136.60,ES,0.161264
1,30.0,89039.54,FI,0.166283
2,48.0,38826.09,IT,0.253933
3,49.0,58823.00,PL,0.246326
4,20.0,86885.48,PL,0.218417


,age,income,country,prediction_score
0,52.0,69374.68,ES,0.596980
1,44.0,54103.40,NEW_COUNTRY,0.352004
2,27.0,42902.94,NL,0.344967
3,18.0,67125.64,IT,0.412471
4,30.0,58969.65,NL,0.263347


Сравним несколько простых показателей до запуска анализатора.

In [3]:
comparison = pd.DataFrame({
    "reference": {
        "age mean": reference_df["age"].mean(),
        "income mean": reference_df["income"].mean(),
        "income missing rate": reference_df["income"].isna().mean(),
        "NEW_COUNTRY rate": reference_df["country"].eq("NEW_COUNTRY").mean(),
        "prediction score mean": reference_df["prediction_score"].mean(),
    },
    "current": {
        "age mean": current_df["age"].mean(),
        "income mean": current_df["income"].mean(),
        "income missing rate": current_df["income"].isna().mean(),
        "NEW_COUNTRY rate": current_df["country"].eq("NEW_COUNTRY").mean(),
        "prediction score mean": current_df["prediction_score"].mean(),
    },
})
display(comparison.round(4))

,reference,current
age mean,39.9513,39.8280
income mean,69578.5045,71386.4250
income missing rate,0.0100,0.0150
NEW_COUNTRY rate,0.0000,0.0105
prediction score mean,0.2861,0.2980


**Ожидание**: age должен остаться стабильным; income – показать умеренный сдвиг; country – заметный сдвиг из-за новой категории. Распределение prediction_score изменится слабо: мы ожидаем warning, если KS-test превысит порог предупреждения, но не достигнет критического порога. Сравним это ожидание со статусами в отчёте.

---

### **Шаг 1**. Профилирование reference-данных

`OfflineWrapper` создаёт анализатор на основе reference-выборки: загружает или генерирует конфигурацию и эталонные статистики признаков. Метод `analyze_df(current_df)` сравнивает с ними текущую выборку и возвращает структурированный отчёт с метриками, порогами и статусами. Дополнительный метод `run_av(current_df)` запускает Adversarial Validation.

OfflineWrapper может использовать готовый YAML-конфиг или создать его автоматически. Через ConfigBuildOptions можно настроить автогенерацию, включая признаки, метрики, пороги и анализ предсказаний. В этом tutorial мы задаём только колонку `prediction_score`; остальные настройки остаются значениями по умолчанию.

Минимальный вызов:

<pre style="margin: 8px 0; padding: 4px 0 4px 14px; border: 0; border-left: 3px solid #888; background: transparent; color: inherit; font-size: 13px; line-height: 1.5;">
analyzer = OfflineWrapper(reference_df)
</pre>

В этом примере данные содержат `prediction_score`. Чтобы анализатор учитывал предсказания, укажем имя этой колонки через `ConfigBuildOptions`:

In [4]:
config_options = ConfigBuildOptions(
    prediction_enabled=True,
    prediction_score_column="prediction_score",
)

In [5]:
analyzer = OfflineWrapper(
    reference_df=reference_df,
    config_options=config_options,
)

### **Шаг 2.1**. Drift report для current-данных.

`analyze_df` проверяет текущую выборку и рассчитывает настроенные метрики относительно reference-профиля. Значения метрик сравниваются с порогами; по их статусам определяются статусы признаков и общий статус анализа. Метод возвращает структурированный Python-словарь report, который можно передать генератору HTML.

При выполнении ячейки также выводятся диагностические сообщения о найденных сдвигах.

Вызов:
<pre style="margin: 8px 0; padding: 4px 0 4px 14px; border: 0; border-left: 3px solid #888; background: transparent; color: inherit; font-size: 13px; line-height: 1.5;">
report = analyzer.analyze_df(current_df)
</pre>

In [6]:
report = analyzer.analyze_df(current_df)

Column 'income': metric 'missing_rate' status=warning (value=0.0150, warning=0.0130, critical=0.0200)
Column 'income': metric 'wasserstein_distance' status=warning (value=1944.3240, warning=1590.6936, critical=1978.1773)
Column 'income': metric 'kstest' status=warning (value=0.0481, warning=0.0400, critical=0.0514)
Column 'income' overall status=warning
Column 'country': metric 'js_divergence' status=warning (value=0.0056, warning=0.0050, critical=0.0200)
Column 'country': metric 'unseen_category_rate' status=warning (value=0.0105, warning=0.0050, critical=0.0200)
Column 'country': metric 'chi2' status=critical (value=0.0000, warning=0.0500, critical=0.0100)
Column 'country': metric 'cramer_v' status=critical (value=0.0948, warning=0.0280, critical=0.0500)
Column 'country': metric 'category_churn' status=critical (value=0.1667, warning=0.0050, critical=0.0200)
Column 'country' overall status=critical
Column 'prediction_score': metric 'kstest' status=warning (value=0.0460, warning=0.040

`analyze_df` возвращает вложенный словарь. На верхнем уровне находятся метаданные и общий статус; в `features` – статусы признаков и результаты отдельных метрик. Каждая метрика содержит значение, пороги `warning` и `critical`, а также свой статус. Ниже посмотрим на общую сводку и один пример, прежде чем передать полный report генератору HTML.

In [7]:
# Общий результат анализа
rich.print({
    "timestamp": report["timestamp"],
    "window_size": report["window_size"],
    "overall_status": report["overall_status"],
    "active_alerts": report["active_alerts"],
})

# Как устроен результат для одного признака и одной метрики
rich.print({
    "feature": "income",
    "feature_status": report["features"]["income"]["status"],
    "js_divergence": report["features"]["income"]["metrics"]["js_divergence"],
})

{'timestamp': '2026-09-26T20:17:43Z', 'window_size': 2000, 'overall_status': 'critical', 'active_alerts': 1}

{
    'feature': 'income',
    'feature_status': 'warning',
    'js_divergence': {
        'value': np.float64(0.0023506079634937086),
        'warning': 0.005,
        'critical': 0.02,
        'status': 'ok'
    }
}

### **Шаг 2.2**. Adversarial Validation report <small>(optional)</small>
Adversarial Validation оценивает разделимость подготовленных датасетов reference и current.

Функция обучает `LightGBM` различать строки двух датасетов, возвращает ROC AUC на out-of-fold предсказаниях и feature importance. Чем выше AUC, тем легче модели различить датасеты. Метрика служит индикатором возможного дрифта.

`analyzer.run_av` возвращает кортеж `(roc_auc, feature_importance DataFrame)`.

Минимальный вызов:
<pre style="margin: 8px 0; padding: 4px 0 4px 14px; border: 0; border-left: 3px solid #888; background: transparent; color: inherit; font-size: 13px; line-height: 1.5;">
av_report = analyzer.run_av(current_df)
</pre>
но колонку `prediction_score` передаем отдельно.

In [8]:
# запуск отчета AV
av_report = analyzer.run_av(
    current_df,
    prediction_col="prediction_score",
)

# вывод отчета AV
rich.print(av_report)

(
    0.513594125,
       feature  importance  importance_std  rank
0   income    0.695452        0.068578     1
1      age    0.242901        0.086141     2
2  country    0.061647        0.027888     3
)

### **Шаг 3**. Готовый HTML-report

`generate_html_report` создаёт HTML-файл из словаря report или сохранённого JSON-отчёта. Можно так же добавить результаты Adversarial Validation. Функция возвращает путь к созданному файлу. 

In [9]:
# название датасета для html-отчета
dataset_name = "Offline drift demo"

# путь к файлу html-отчета
html_report_path = Path("../reports/offline_drift_report.html")

generated_html = generate_html_report(
    report=report,
    av_report=av_report,
    dataset_name=dataset_name,
    output_path=html_report_path,
)

`display_html_report` принимает путь к созданному HTML-файлу и показывает отчёт прямо в ноутбуке.

In [ ]:
display_html_report(generated_html, height=1300)